# Running the Book in Google Colab
stough 202-

[Colab](https://colab.research.google.com/) runs notebooks in the cloud, so you don't need to install anything, and it offers a free GPU for the neural network chapters. But the book's notebooks are not standalone: they load images from `dip_pics/` and helper code from `dip_utils/`, through relative paths like `../dip_utils`. A notebook opened by itself in Colab has none of that.

This notebook sets you up properly. Run it once (and again whenever you want the latest version of the book):

1. It copies the whole book (a `git clone`) into your **Google Drive**, so the book and your work in it survive after the Colab session ends.
2. It prepares a folder in your Drive for the datasets that a few chapters use.
3. It prints a short setup cell that you paste at the top of any book notebook you open in Colab.

If you're working locally (VS Code or JupyterLab), you don't need this; see the [README](./README.md) instead.

## 1. Settings

If you made a [private fork](./private_fork_instructions.md) of the book, put its address in `REPO_URL`. A private repository needs a GitHub [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) with read access to it. Don't paste the token into this notebook: add it as a Colab **secret** (the key icon in the left sidebar), and put the secret's name in `TOKEN_SECRET`.

In [ ]:
REPO_URL = 'https://github.com/profstough/imageprocessing-awesome.git'   # or your private fork
TOKEN_SECRET = None      # e.g. 'GITHUB_TOKEN', the name of a Colab secret, for a private fork
FOLDER = 'imageprocessing-awesome'   # the folder the book goes in, inside My Drive

## 2. Copy the book into your Drive

Colab will ask for permission to use your Google Drive. If the book is already there from an earlier run, this updates it instead (`git pull`); your own changes to the notebooks are kept, but if they conflict with an update, git will say so and leave everything as it was.

In [ ]:
import os
import subprocess

try:
    from google.colab import drive
except ImportError:
    raise RuntimeError('This notebook is meant for Google Colab. Locally, follow the README instead.')

drive.mount('/content/drive')
BOOK = f'/content/drive/MyDrive/{FOLDER}'
DATA = '/content/drive/MyDrive/dip_data'

url = REPO_URL
if TOKEN_SECRET:
    from google.colab import userdata
    url = REPO_URL.replace('https://', f'https://{userdata.get(TOKEN_SECRET)}@')

if os.path.isdir(os.path.join(BOOK, '.git')):
    print('The book is already in your Drive; updating it.')
    result = subprocess.run(['git', '-C', BOOK, 'pull', '--ff-only'], capture_output=True, text=True)
else:
    print('Copying the book into your Drive (a minute or two)...')
    result = subprocess.run(['git', 'clone', url, BOOK], capture_output=True, text=True)
print((result.stdout + result.stderr).replace(url, REPO_URL))   # never print a token
assert result.returncode == 0, 'git did not succeed; see its message above.'
print('The book is in', BOOK)

## 3. Datasets and packages

Colab already has NumPy, Matplotlib, scikit-image, scikit-learn, SciPy, OpenCV and PyTorch. It lacks [ipympl](https://matplotlib.org/ipympl/), which draws the book's interactive (`%matplotlib widget`) figures; the setup cell below installs it in each session.

The MNIST data downloads itself the first time it's used. The ORL and Yale face datasets have to be downloaded by hand: see the dataset table in the [README](./README.md#datasets), and put them in the `dip_data` folder this cell creates in your Drive (e.g. `dip_data/ORL/`).

In [ ]:
os.makedirs(DATA, exist_ok=True)
print('Put datasets in', DATA)

## 4. Open a chapter

In Colab, choose **File → Open notebook → Google Drive**, and open any notebook from the `imageprocessing-awesome` folder, for example `Color/color_HSV.ipynb`. Then, before anything else, insert a code cell at the very top of that notebook, paste in the cell printed below, set `CHAPTER` to the notebook's folder, and run it. It moves into that folder (so `../dip_utils` and `../dip_pics` are found), tells the book where your datasets are, and turns on interactive figures.

For the neural network chapters, also choose **Runtime → Change runtime type → GPU**.

In [ ]:
print(f'''# --- Colab setup: run this first. Set CHAPTER to this notebook's folder. ---
CHAPTER = 'Color'
from google.colab import drive, output
drive.mount('/content/drive')
%cd {BOOK}/{{CHAPTER}}
%env DIP_DATA={DATA}
!pip -q install ipympl
output.enable_custom_widget_manager()''')

Your work is saved in the notebooks in your Drive. Remember that if you're in a course using this book, the activities are there to help you learn: see the note on AI in the [preface](./TOC.ipynb).